# FM2Edge Phase 1.5 — Oxford-IIIT Pet Colab acceptance
This notebook validates download → breed-domain split → PIDNet-S training → held-out-domain evaluation. Oxford-specific code is disposable and does not enter the core pipeline.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Colab menu: Runtime > Change runtime type > GPU"

Set your Git remote below. Use a private repository if preferred.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/YOUR_ACCOUNT/FM2Edge.git"  # EDIT
BRANCH = "main"
assert "YOUR_ACCOUNT" not in REPO_URL, "Edit REPO_URL first"
if not os.path.isdir("/content/FM2Edge/.git"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "/content/FM2Edge"], check=True)
else:
    subprocess.run(["git", "-C", "/content/FM2Edge", "pull", "--ff-only"], check=True)

In [ ]:
%cd /content/FM2Edge
%pip install -q -e ".[tensorboard]"

In [ ]:
!python scripts/download_oxford_pet.py --root data/oxford_pet
!python scripts/prepare_oxford_pet.py --root data/oxford_pet --output data/oxford_pet/manifest_smoke.csv --max-breeds 12 --max-samples-per-breed 40 --seed 42
!python scripts/make_splits.py --manifest data/oxford_pet/manifest_smoke.csv --output-dir configs/splits/oxford_pet_smoke --folds 5 --seed 42

In [ ]:
!python scripts/train.py --config configs/experiments/pidnet_s_oxford_pet_smoke.yaml
!python scripts/evaluate.py --config configs/experiments/pidnet_s_oxford_pet_smoke.yaml

Optional: persist only the self-contained run artifacts to Google Drive.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    source = "/content/FM2Edge/results/pidnet_s_oxford_pet_smoke"
    destination = "/content/drive/MyDrive/FM2Edge_results/pidnet_s_oxford_pet_smoke"
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("saved:", destination)